In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)
from sklearn.utils.class_weight import compute_class_weight

In [6]:
# 1. CONFIGURATION


DATA_DIR = os.environ.get("CHEST_XRAY_DATA_DIR", r"E:\Pneumonia")
DATA_DIR = os.path.abspath(DATA_DIR)
OUTPUT_DIR = os.path.join(DATA_DIR, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

required_dirs = [
    os.path.join(DATA_DIR, split, class_name)
    for split in ("train", "val", "test")
    for class_name in ("NORMAL", "PNEUMONIA")
]
missing_dirs = [directory for directory in required_dirs if not os.path.isdir(directory)]
if missing_dirs:
    raise FileNotFoundError(
        "Dataset must contain train/val/test folders with NORMAL and PNEUMONIA subfolders. "
        f"Missing: {missing_dirs}"
    )

IMG_SIZE = (150, 150)
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 1e-3
RANDOM_SEED = 42

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Dataset root:", DATA_DIR)
print("Output folder:", OUTPUT_DIR)

Dataset root: E:\Pneumonia
Output folder: E:\Pneumonia\output


In [7]:
# 3. DATA LOADING


train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    brightness_range=[0.85, 1.15],
    horizontal_flip=True
)

eval_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(DATA_DIR, "train"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    classes=["NORMAL", "PNEUMONIA"],
    shuffle=True,
    seed=RANDOM_SEED
)

val_generator = eval_datagen.flow_from_directory(
    os.path.join(DATA_DIR, "val"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    classes=["NORMAL", "PNEUMONIA"],
    shuffle=False
)

test_generator = eval_datagen.flow_from_directory(
    os.path.join(DATA_DIR, "test"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    classes=["NORMAL", "PNEUMONIA"],
    shuffle=False
)

print("\nClass mapping:", train_generator.class_indices)
print("Training images   :", train_generator.samples)
print("Validation images :", val_generator.samples)
print("Test images       :", test_generator.samples)



Found 5216 images belonging to 2 classes.
Found 16 images belonging to 2 classes.
Found 624 images belonging to 2 classes.

Class mapping: {'NORMAL': 0, 'PNEUMONIA': 1}
Training images   : 5216
Validation images : 16
Test images       : 624


In [8]:
# 4. EXPLORATORY DATA ANALYSIS (EDA)



print("EDA – Generating visualisations")



eda_datagen = ImageDataGenerator(rescale=1.0/255)  # no augmentation, just rescale

eda_train_gen = eda_datagen.flow_from_directory(
    os.path.join(DATA_DIR, "train"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    classes=["NORMAL", "PNEUMONIA"],
    shuffle=True,
    seed=RANDOM_SEED
)

# ---- For augmentation preview, 
preview_datagen = ImageDataGenerator(
    # NO rescale here – we'll pass raw 0–255 images
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    brightness_range=[0.85, 1.15],
    horizontal_flip=True
)


# 4a. Class distribution (with percentages)

def plot_class_distribution():
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    splits = ['train', 'val', 'test']
    for ax, split in zip(axes, splits):
        normal_dir = os.path.join(DATA_DIR, split, "NORMAL")
        pneumonia_dir = os.path.join(DATA_DIR, split, "PNEUMONIA")
        n_normal = len(os.listdir(normal_dir)) if os.path.exists(normal_dir) else 0
        n_pneumonia = len(os.listdir(pneumonia_dir)) if os.path.exists(pneumonia_dir) else 0
        total = n_normal + n_pneumonia
        ax.bar(["NORMAL", "PNEUMONIA"], [n_normal, n_pneumonia],
               color=["#1f77b4", "#ff7f0e"])
        ax.set_title(f"{split.capitalize()} set\n(total: {total})")
        ax.set_ylabel("Number of images")
        for i, v in enumerate([n_normal, n_pneumonia]):
            pct = v / total * 100 if total > 0 else 0
            ax.text(i, v + 5, f"{v}\n({pct:.1f}%)", ha='center', va='bottom')
    plt.suptitle("Class Distribution Across Splits", fontsize=16)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "class_distribution.png"), dpi=150)
    plt.close()
    print(" - Saved class_distribution.png")


# 4b. Sample images (using eda_train_gen)

def plot_sample_images():
    sample_x, sample_y = next(eda_train_gen)   # uses EDA generator
    normal_idx = np.where(sample_y == 0)[0]
    pneumonia_idx = np.where(sample_y == 1)[0]
    n = min(10, len(normal_idx), len(pneumonia_idx))
    selected_normal = normal_idx[:n]
    selected_pneumonia = pneumonia_idx[:n]
    
    fig, axes = plt.subplots(2, n, figsize=(2*n, 4))
    if n == 0:
        print("Warning: Not enough samples in batch.")
        return
    for i, idx in enumerate(selected_normal):
        axes[0, i].imshow(sample_x[idx])
        axes[0, i].set_title("NORMAL")
        axes[0, i].axis('off')
    for i, idx in enumerate(selected_pneumonia):
        axes[1, i].imshow(sample_x[idx])
        axes[1, i].set_title("PNEUMONIA")
        axes[1, i].axis('off')
    plt.suptitle("Sample Training Images (NORMAL top, PNEUMONIA bottom)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "sample_images.png"), dpi=150)
    plt.close()
    print(" - Saved sample_images.png")


# 4c. Pixel intensity histograms (using eda_train_gen)

def plot_pixel_histograms():
    x_normal = []
    x_pneumonia = []
    # Collect a few batches
    for _ in range(3):
        x_batch, y_batch = next(eda_train_gen)
        x_normal.extend(x_batch[y_batch == 0])
        x_pneumonia.extend(x_batch[y_batch == 1])
    normal_pixels = np.concatenate([img.flatten() for img in x_normal])
    pneumonia_pixels = np.concatenate([img.flatten() for img in x_pneumonia])

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].hist(normal_pixels, bins=50, alpha=0.7, color='blue', density=True)
    axes[0].set_title("NORMAL – Pixel Intensity")
    axes[0].set_xlabel("Pixel value")
    axes[1].hist(pneumonia_pixels, bins=50, alpha=0.7, color='orange', density=True)
    axes[1].set_title("PNEUMONIA – Pixel Intensity")
    axes[1].set_xlabel("Pixel value")
    plt.suptitle("Pixel Intensity Comparison")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "pixel_histograms.png"), dpi=150)
    plt.close()
    print(" - Saved pixel_histograms.png")


# 4d. Mean & Std images per class (using eda_train_gen)

def plot_mean_std_images():
    n_samples = 500
    x_normal, x_pneumonia = [], []
    count_n, count_p = 0, 0
    while count_n < n_samples or count_p < n_samples:
        x_batch, y_batch = next(eda_train_gen)
        x_normal.extend(x_batch[y_batch == 0])
        x_pneumonia.extend(x_batch[y_batch == 1])
        count_n = len(x_normal)
        count_p = len(x_pneumonia)
    x_normal = np.array(x_normal[:n_samples])
    x_pneumonia = np.array(x_pneumonia[:n_samples])
    
    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    axes[0, 0].imshow(x_normal.mean(axis=0))
    axes[0, 0].set_title("Mean NORMAL")
    axes[0, 0].axis('off')
    axes[0, 1].imshow(x_pneumonia.mean(axis=0))
    axes[0, 1].set_title("Mean PNEUMONIA")
    axes[0, 1].axis('off')
    axes[1, 0].imshow(x_normal.std(axis=0), cmap='hot')
    axes[1, 0].set_title("Std Dev NORMAL")
    axes[1, 0].axis('off')
    axes[1, 1].imshow(x_pneumonia.std(axis=0), cmap='hot')
    axes[1, 1].set_title("Std Dev PNEUMONIA")
    axes[1, 1].axis('off')
    plt.suptitle("Mean and Standard Deviation Images per Class")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "mean_std_images.png"), dpi=150)
    plt.close()
    print(" - Saved mean_std_images.png")


# 4e. Brightness vs Contrast (using eda_train_gen)

def plot_brightness_contrast():
    x_batch, y_batch = next(eda_train_gen)
    brightness = x_batch.mean(axis=(1,2,3))
    contrast = x_batch.std(axis=(1,2,3))
    plt.figure(figsize=(8, 6))
    for c, color, label in zip([0, 1], ['#1f77b4', '#ff7f0e'], ['NORMAL', 'PNEUMONIA']):
        mask = (y_batch == c)
        plt.scatter(brightness[mask], contrast[mask], c=color, label=label, alpha=0.7)
    plt.xlabel("Brightness (mean pixel)")
    plt.ylabel("Contrast (std pixel)")
    plt.title("Brightness vs Contrast per Image")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "brightness_contrast.png"), dpi=150)
    plt.close()
    print(" - Saved brightness_contrast.png")


# 4f. Augmentation preview 

def plot_augmented_samples():
    # Get one image from the EDA generator (already in 0–1 range)
    orig_img, _ = next(eda_train_gen)
    orig_img = orig_img[0]   # shape (150,150,3)

    # Convert back to 0–255 (because preview_datagen has NO rescale)
    orig_img_raw = (orig_img * 255).astype(np.uint8)
    single_img = np.expand_dims(orig_img_raw, axis=0)
    dummy_label = np.array([0])

    # Use the dedicated preview generator (no rescale)
    aug_gen = preview_datagen.flow(single_img, dummy_label, batch_size=1)

    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    # Original
    axes[0, 0].imshow(orig_img_raw)   # show raw 0–255
    axes[0, 0].set_title("Original")
    axes[0, 0].axis('off')
    # Augmented versions
    for i in range(1, 8):
        aug_img, _ = next(aug_gen)
        row = 0 if i < 4 else 1
        col = i % 4
        axes[row, col].imshow(aug_img[0].astype(np.uint8))
        axes[row, col].set_title(f"Aug {i}")
        axes[row, col].axis('off')
    plt.suptitle("Augmentation Examples")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "augmented_samples.png"), dpi=150)
    plt.close()
    print(" - Saved augmented_samples.png")




plot_class_distribution()
plot_sample_images()
plot_pixel_histograms()
plot_mean_std_images()
plot_brightness_contrast()
plot_augmented_samples()

print("EDA plots saved in:", OUTPUT_DIR)


EDA – Generating visualisations
Found 5216 images belonging to 2 classes.
 - Saved class_distribution.png
 - Saved sample_images.png
 - Saved pixel_histograms.png
 - Saved mean_std_images.png
 - Saved brightness_contrast.png
 - Saved augmented_samples.png
EDA plots saved in: E:\Pneumonia\output


In [4]:


# 5. CLASS WEIGHTS


classes = np.unique(train_generator.classes)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_generator.classes
)
class_weight_dict = {int(k): float(v) for k, v in zip(classes, class_weights)}
print("\nClass weights:", class_weight_dict)




Class weights: {0: 1.9448173005219984, 1: 0.6730322580645162}


In [5]:

# 6. BUILD CNN MODEL


def build_cnn_model(input_shape, learning_rate=LEARNING_RATE):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, 3, activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),

        layers.Conv2D(64, 3, activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),

        layers.Conv2D(128, 3, activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),

        layers.Conv2D(256, 3, activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),

        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
    )
    return model

model = build_cnn_model(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
model.summary()



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 150, 150, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 150, 150, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 75, 75, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 75, 75, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 75, 75, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 37, 37, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 37, 37, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 37, 37, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 18, 18, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 18, 18, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 18, 18, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 9, 9, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 423,361 (1.61 MB)

 Trainable params: 422,401 (1.61 MB)

 Non-trainable params: 960 (3.75 KB)

In [11]:

# 7. CALLBACKS


checkpoint_path = os.path.join(OUTPUT_DIR, "chest_xray_cnn_best.keras")

callbacks_list = [
    callbacks.EarlyStopping(
        monitor="val_loss", patience=6,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5,
        patience=3, min_lr=1e-6, verbose=1
    ),
    callbacks.ModelCheckpoint(
        checkpoint_path, save_best_only=True,
        monitor="val_loss", verbose=1
    )
]


In [12]:


# 8. TRAIN



print("TRAINING CNN")


history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=callbacks_list,
    verbose=1
)



TRAINING CNN


d:\Programs\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 704ms/step - accuracy: 0.8098 - auc: 0.8945 - loss: 0.4041
Epoch 1: val_loss improved from None to 5.68302, saving model to output\chest_xray_cnn_best.keras
163/163 ━━━━━━━━━━━━━━━━━━━━ 117s 708ms/step - accuracy: 0.8535 - auc: 0.9318 - loss: 0.3321 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 5.6830 - learning_rate: 0.0010
Epoch 2/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 749ms/step - accuracy: 0.8896 - auc: 0.9534 - loss: 0.2691
Epoch 2: val_loss improved from 5.68302 to 1.32061, saving model to output\chest_xray_cnn_best.keras
163/163 ━━━━━━━━━━━━━━━━━━━━ 122s 751ms/step - accuracy: 0.8930 - auc: 0.9555 - loss: 0.2655 - val_accuracy: 0.5625 - val_auc: 0.6797 - val_loss: 1.3206 - learning_rate: 0.0010
Epoch 3/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 730ms/step - accuracy: 0.9147 - auc: 0.9696 - loss: 0.2164
Epoch 3: val_loss did not improve from 1.32061
163/163 ━━━━━━━━━━━━━━━━━━━━ 119s 731ms/step - accuracy: 0.9158 - auc: 0.9723 - loss: 0.2062 - va

In [13]:


# 9. TEST EVALUATION



print("TEST RESULTS")


test_generator.reset()
predictions = model.predict(test_generator, verbose=1)
y_pred = (predictions.ravel() > 0.5).astype(int)
y_true = test_generator.classes

print("\nClassification Report:")
print(
    classification_report(
        y_true, y_pred,
        target_names=["NORMAL", "PNEUMONIA"],
        digits=4
    )
)

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

auc = roc_auc_score(y_true, predictions.ravel())
test_accuracy = np.mean(y_true == y_pred)

print(f"\nROC AUC: {auc:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

pneumonia_recall = cm[1, 1] / (cm[1, 0] + cm[1, 1])
print(f"PNEUMONIA Recall (sensitivity): {pneumonia_recall:.4f}")


TEST RESULTS


d:\Programs\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 267ms/step

Classification Report:
              precision    recall  f1-score   support

      NORMAL     0.9630    0.1111    0.1992       234
   PNEUMONIA     0.6516    0.9974    0.7882       390

    accuracy                         0.6651       624
   macro avg     0.8073    0.5543    0.4937       624
weighted avg     0.7684    0.6651    0.5674       624

Confusion Matrix:
[[ 26 208]
 [  1 389]]

ROC AUC: 0.8347
Test Accuracy: 0.6651
PNEUMONIA Recall (sensitivity): 0.9974


In [14]:


# 10. TRAINING CURVES (saved to output/)


plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Val")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Val")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(history.history["auc"], label="Train")
plt.plot(history.history["val_auc"], label="Val")
plt.title("AUC")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=300, bbox_inches="tight")
plt.close()

print("\nTraining curves saved to:", os.path.join(OUTPUT_DIR, "training_curves.png"))
print(f"Best model saved to: {checkpoint_path}")
print("\nAll output files are in:", OUTPUT_DIR)
print("\nDone.")


Training curves saved to: output\training_curves.png
Best model saved to: output\chest_xray_cnn_best.keras

All output files are in: output

Done.
